# Deep Learning 002 — Machine Learning vs Deep Learning

The difference is not "one is better". It is **who invents the features.** We run the
same task both ways on two datasets — one where the boundary is curved and one where it
is straight — and let the results decide.

Every number below is measured with stratified cross-validation, because a single
train/test split on a few hundred rows is too noisy to draw a conclusion from.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

# concertriccir2.csv has no header: x1, x2, label
c = pd.read_csv('../data/concertriccir2.csv', header=None, names=['x1', 'x2', 'y'])
X, y = c[['x1', 'x2']].to_numpy(), c['y'].to_numpy().astype(int)
print(c.shape, '| class balance:', np.bincount(y))

fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.scatter(X[y == 0, 0], X[y == 0, 1], marker='o', s=14, label='0')
ax.scatter(X[y == 1, 0], X[y == 1, 1], marker='^', s=14, label='1')
ax.set_title('concertriccir2.csv - rings, not sides'); ax.legend()
plt.tight_layout(); plt.show()

> **Note on the folds.** This file is sorted by class. A plain `cross_val_score` would
> hand some folds a single label and report accuracies *below chance*. `StratifiedKFold`
> with `shuffle=True` fixes it — worth remembering the first time a CV score comes back
> at 40% on balanced data.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=0)

def score(model, Z):
    s = cross_val_score(model, Z, y, cv=cv)
    return s.mean(), s.std()

## Round 1 — plain logistic regression on the raw two columns

A straight line has to separate two nested rings. It cannot — and it scores *below
guessing*, because the best line it can find is actively misleading.

In [ ]:
m, sd = score(LogisticRegression(max_iter=3000), X)
print(f'raw logistic regression        {m:.1%} +/- {sd:.1%}')
print(f'chance on balanced classes     50.0%')

## Round 2 — the same model, after *you* invent the features

This is classical machine learning: a simple model where **the intelligence is in the
feature engineering.** Add squared and interaction terms so the linear model can express
a curve.

In [ ]:
def expand(A):
    x1, x2 = A[:, 0], A[:, 1]
    return np.column_stack([x1, x2, x1**2, x2**2, x1 * x2])

m, sd = score(LogisticRegression(max_iter=3000), expand(X))
print(f'logistic + x1^2, x2^2, x1*x2   {m:.1%} +/- {sd:.1%}')
print()
print('Better than chance now, but not good. Those features are a GUESS about the')
print('shape of the boundary, and this guess is only partly right. That is the')
print('failure mode of feature engineering: you have to already know the answer.')

## Round 3 — a network on the raw columns, no feature engineering at all

In [ ]:
for hidden in [(8,), (32, 16)]:
    m, sd = score(MLPClassifier(hidden_layer_sizes=hidden, max_iter=6000,
                                random_state=0), X)
    print(f'MLP {str(hidden):<10} on RAW x1, x2   {m:.1%} +/- {sd:.1%}')
print()
print('features supplied by you: none')

In [ ]:
xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-.5, X[:, 0].max()+.5, 250),
                     np.linspace(X[:, 1].min()-.5, X[:, 1].max()+.5, 250))
grid = np.c_[xx.ravel(), yy.ravel()]
fitted = [('raw logistic', LogisticRegression(max_iter=3000).fit(X, y), lambda A: A),
          ('logistic + hand features',
           LogisticRegression(max_iter=3000).fit(expand(X), y), expand),
          ('MLP(32,16) on raw columns',
           MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=6000,
                         random_state=0).fit(X, y), lambda A: A)]

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.4))
for ax, (name, mdl, tf) in zip(axes, fitted):
    ax.contourf(xx, yy, mdl.predict(tf(grid)).reshape(xx.shape), alpha=0.25, levels=1)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], marker='o', s=10)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], marker='^', s=10)
    ax.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

## Does the network need more data to win? Measure it rather than assuming.

The standard claim is that deep learning only pays off once the dataset is large.
Subsample and re-score at each size.

In [ ]:
rng = np.random.default_rng(0)
cv4 = StratifiedKFold(4, shuffle=True, random_state=0)
print(f"{'n rows':>7}{'logistic+hand':>16}{'MLP(32,16)':>13}")
for n in (25, 50, 100, 200, 350, 500):
    idx = rng.permutation(len(X))[:n]
    a = cross_val_score(LogisticRegression(max_iter=3000),
                        expand(X[idx]), y[idx], cv=cv4).mean()
    b = cross_val_score(MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=6000,
                                      random_state=0), X[idx], y[idx], cv=cv4).mean()
    print(f'{n:>7}{a:>16.1%}{b:>13.1%}')

**The expected result did not appear, and that is worth stopping on.**

The network wins at *every* size, including 25 rows — and the gap **widens** with data
rather than opening up only at the end. So "deep learning needs a lot of data before it
is worth it" is not what this experiment shows.

The reason is that this comparison is not deep-versus-shallow, it is **right model class
versus wrong model class.** The hand-engineered features are a bad guess about this
boundary, so the linear model is capped no matter how much data it receives. More data
cannot rescue a model that cannot express the answer.

To see the other side of the trade you need a problem where the simple model is *right*.

## Where the simple model wins

`placement.csv` has a boundary that genuinely is close to a straight line.

In [ ]:
d = pd.read_csv('../data/placement.csv')
Xp, yp = d[['cgpa', 'resume_score']].to_numpy(), d['placed'].to_numpy().astype(int)
print(f"{'n rows':>7}{'logistic':>11}{'MLP(32,16)':>13}")
for n in (25, 50, 100):
    idx = rng.permutation(len(Xp))[:n]
    a = cross_val_score(LogisticRegression(max_iter=3000),
                        Xp[idx], yp[idx], cv=cv4).mean()
    b = cross_val_score(MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=6000,
                                      random_state=0), Xp[idx], yp[idx], cv=cv4).mean()
    print(f'{n:>7}{a:>11.1%}{b:>13.1%}')

Here logistic regression wins at every size, and by most at the smallest. The network
has to *learn* that the boundary is a straight line; the linear model was told.

**The trade, stated plainly.** Classical ML asks you to know which features matter and
rewards you with a model that works on small data — *when your guess is right*. Deep
learning asks you for data and rewards you by finding the features itself — *when your
guess would have been wrong*. Neither is the better answer in general. The question is
always whether you already know the shape of the problem.

## Exercises

1. Find hand-engineered features that actually suit rings (hint: what is constant along
   a circle?). Does logistic regression catch the network once you give it the right
   feature?
2. Re-run the sweep with `hidden_layer_sizes=(2,)`. Where does the network now lose, and
   why?
3. Flip 10% of the labels in `placement.csv`. Which model degrades faster?
4. Re-run Round 1 with a plain `cross_val_score(..., cv=5)` and reproduce the
   below-chance score. Explain it.